In [ ]:
import numpy as np
import matplotlib.pyplot as plt

a = 1
b = 1
lim = 1.6

x = np.linspace(-lim,lim,100)


V = 1/2 * (a) * x**2 + b/4 * x**4

fig, ax = plt.subplots(1,1)
ax.set_xlabel(r'$\phi$')
ax.set_ylabel(r'$V_\text{eff}$')

# Turn off tick labels
ax.set_yticklabels([])
ax.set_xticklabels([])
ax.set_xticks([])
ax.set_yticks([])
ax.spines['bottom'].set_position('zero')
ax.spines['left'].set_position('zero')



ax.plot(x,V, 'b--')

c = -1

V = 1/2 * c * x**2 + b/4 * x**4

ax.plot(x,V, 'orange')

In [ ]:
'''
This code builds the analytical function above which results from applying the thin-
wall approximation to the symmetron model.

Then plots the function with respect to (gamma x s).

The result concurrs with that found in the paper linked above.
'''

import numpy as np
import matplotlib.pyplot as plt

def analytical_eq(s):
    '''
    Builds the analytical equation found by applying thin-wall
    approximation to symmetron model

    Variables:
    s: Array, domain of the analytical function

    Outputs:
    phi: Array, function array resulting from operation on variable s
    '''

    p = 25 # Phase parameter
    g = np.sqrt(2*(p - 1)) # The effective mass of interior scalar fluctuations in units of γ
    x0 = (2 + g**2)**(-1/2) # Field boundary value
    pve_step = np.heaviside(s, 0) # Step function Theta(s) to isolate exterior
    nve_step = np.heaviside(-s, 0) # Step function Theta(-s) to isolate interior


    a = np.asinh(g/x0) - g*s
    interior = nve_step * g * (1/np.sinh(a))
    exterior = pve_step * np.tanh(s + np.atanh(x0)) # Exterior term of equation

    phi = (interior + exterior)

    return phi

def plot(s, phi):
    '''
    Creates figure, plots two arrays and annotates.

    Variables:
    s: Array
    phi: Array

    Outputs:
    None
    '''

    fig, ax = plt.subplots()
    ax.set_xlabel(r'$\gamma(r-R)$')
    ax.set_ylabel(r'$\phi / v$')

    plt.plot(s,phi)
    plt.show

    return None

def main():

    s = np.linspace(-5,5,100)

    phi = analytical_eq(s)
    plot(s, phi)

    return

ana = main()

print(ana)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
                                                                                
def get_A(n, h, x):
    A = np.zeros((n+1,n+1))
    A[0,0] = 1
    A[n,n] = 1
    K = 2
    R = 0
    for i in range(1, n):
        A[i,i-1] = 1/(h**2) - 1/((x[i])*h)
        A[i,i] = -2/(h**2) - K
        A[i,i+1] = 1/(h**2) + 1/((x[i])*h)

    return A

def get_B(n, h, x, chi0, v):
    B = np.zeros(n+1)

    B[-1] = 1


    return B

def create_fig(xlabel, ylabel, title, n):

    fig, ax = plt.subplots()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    #ax.text(0.15, 144, f'n = {n}')

    return fig

def plot_data(xlabel, ylabel, title, n, x0, xn, sol):

    create_fig(xlabel, ylabel, title, n)
    x = np.linspace(x0, xn, n+1)
    plt.plot(x,sol)
    plt.show()

def main():

    n = 100
    p = 25 # Phase parameter
    g = np.sqrt(2*(p - 1)) # The effective mass of interior scalar fluctuations in units of γ
    chi0 = (2 + g**2)**(-1/2) # Field boundary value
    x0 = 1e-9
    xn = 50
    x = np.linspace(x0,xn, n)
    v = 1
    h = xn-x0/n
    
    A = get_A(n, h, x)
    B = get_B(n, h, x, chi0, v)

    print(B)

    y = np.linalg.solve(A,B)

    plot_data('r', 'f/v', 'Exterior Solution', n, x0, xn, y)

    return None

print(main())

In [ ]:
from autograd import *
from autograd.numpy import *
from pylab import *

def Fn(x,y,n):
    h = (x[-1] - x[0]) / (n-1)
    ypp = (y[2:] - 2*y[1:-1] + y[:-2]) / h**2
    yp = (y[2:] - y[:-2]) / h
    return ypp + (2/x[1:-1])*yp - 24*y[1:-1] - (y[1:-1])**3

n = 50
dom = (1e-9,10)
bv = 1/sqrt(50)
ran = (0,bv)
x = linspace(dom[0],dom[1],n)
yg = linspace(ran[0],ran[1],n)
yn = yg.copy()
N_max = 30

J = jacobian(lambda y: Fn(x, y, n))

for i in range(N_max):
    Fy = Fn(x,yg,n)

    Jy = J(yg)

    delta_y = linalg.solve(Jy[:,1:-1], -Fy)
    yn[1:-1] = yg[1:-1] + delta_y
    if linalg.norm(delta_y) < 1e-4:
        print(f"Converged in {i+1} iterations")
        break
    else:
        yg = yn

plot(x,yn,label='Newton-Raphson')
title('Solution')
xlabel('x')
ylabel('y')

num = yn

print(num)


In [ ]:
'''
NB: Example of type of code used, not usable in this notebook
'''

print(ana)
print(num)

x = linspace(1e-9,10,50)
er = abs((ana - num) / num)

plot(x,er,label='Error-Introduced')
title('Error Introduced (Interior)')
xlabel('x')
ylabel('Error (%)')